In [ ]:
# Install required packages
!pip install -qqq "stable-baselines3[extra]" gymnasium matplotlib pygame ale-py

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
import time

In [ ]:
env = gym.make('LunarLander-v3')

model = PPO(
    policy = "MlpPolicy",
    env = env,
    verbose = 1,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.99,
    learning_rate = 2.5e-4,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    device = "mps"
)

In [ ]:
model.learn(total_timesteps=1E6)
model_name = "RyanModelV2"
model.save(model_name)

In [ ]:
# Load your trained PPO model
model = PPO.load("RyanModelV2")  # Load from the extracted policy.pth

print("Model loaded successfully!")
print(f"Model action space: {model.action_space}")
print(f"Model observation space: {model.observation_space}")

In [ ]:
env = gym.make('LunarLander-v3', render_mode='human')

def visualize_agent_realtime(episodes=3, delay=0.01):
    """Visualize the agent playing in real-time"""
    print(f"Starting visualization with environment: {env.spec.id}")
    
    for episode in range(episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        total_reward = 0
        step_count = 0
        
        print(f"\n=== Episode {episode + 1} ===")
        
        while not done and not truncated:
            # Get action from the trained model
            action, _ = model.predict(obs, deterministic=True)
            
            # Take action in environment
            obs, reward, done, truncated, info = env.step(action)
            total_reward += reward
            step_count += 1
            
            # Render the environment
            try:
                env.render()
            except Exception as e:
                print(f"Render error: {e}")
                break
            
            # Slow down for viewing
            time.sleep(delay)
            
            # Safety limit to prevent infinite loops
            if step_count > 1000:
                print("Step limit reached, ending episode")
                break
            
        print(f"Episode {episode + 1} finished with reward: {total_reward}, steps: {step_count}")
    
    env.close()
    print("Visualization complete!")

# Run the visualization now
visualize_agent_realtime(episodes=2, delay=0.05)

### Q-learning

In [ ]:
import gymnasium as gym 
import random 
import numpy as np 

In [ ]:
env = gym.make("FrozenLake-v1", map_name = "4x4", is_slippery = False, render_mode="rgb_array")

In [ ]:
env.observation_space

In [ ]:
env.action_space

In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n
print(f"State space: {state_space}")
print(f"Action space: {action_space}")

In [ ]:
def q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

q = q_table(state_space, action_space)

In [ ]:
def greedy_policy(state, q_table):
    '''
    state: current state
    q_table: Q-table
    returns action with highest Q-value for the given state
    '''
    action = np.argmax(q_table[state][:])
    return action

In [ ]:
import random 

def epsilon_greedy(epislon, Q, state):
    if random.random() < epislon:
        return random.randint(0, 3)
    else:
        return greedy_policy(state, Q)
    

In [ ]:
def update_q_table(Q, state, action, reward, next_state, gamma, lr):
    Q[state][action] += lr * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
    return Q

In [ ]:
n_training_steps = 100000
lr = 0.2
n_evaluations = 100
max_steps = 99 
gamma = 0.95 
epsilon_decay = 0.999
epsilon = 1.0

In [ ]:
from tqdm.auto import tqdm

def train_loop(episodes, lr, gamma, max_steps, epsilon, epsilon_decay, q_table):
    for episode in tqdm(range(episodes)):
        state, info = env.reset()
        
        for step in range(max_steps):
            action = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(action) 
            
            q_table = update_q_table(q_table, state, action, reward, next_state, gamma, lr)
            state = next_state
            if step % 10 == 0:
                print(f"Episode: {episode}, Step: {step}, Epsilon: {epsilon}, Reward: {reward}")

            if done or truncated:
                break

        epsilon *= epsilon_decay
    return q_table

In [ ]:
from tqdm.auto import tqdm

def train_loop(episodes, lr, gamma, max_steps, epsilon, epsilon_decay, q_table):
    reward_list = []
    for episode in tqdm(range(episodes)):
        state, info = env.reset()

        total_reward = 0
        for step in range(max_steps):
            action = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(action) 
            
            # Always update Q-table, even when episode ends
            q_table = update_q_table(q_table, state, action, reward, next_state, gamma, lr)
            state = next_state

            total_reward += reward
            if done or truncated:
                total_reward += reward 
                reward_list.append(total_reward)
                break
        
        epsilon *= epsilon_decay
    return q_table, reward_list

In [ ]:
results = train_loop(n_training_steps, lr, gamma, max_steps, epsilon, epsilon_decay, q)

In [ ]:
updated_q_table, reward_list = results
np.mean(reward_list), np.std(reward_list)

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="human")

def watch_agent_play(episodes=3):
    for episode in range(episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        
        print(f"\n=== Episode {episode + 1} ===")
        
        while not done:
            # Get action from trained Q-table (no exploration)
            action = greedy_policy(state, updated_q_table)
            
            # Take action
            next_state, reward, done, truncated, info = env.step(action)
            
            total_reward += reward
            state = next_state
            
            # The show the window 
            env.render()
            
            if done or truncated:
                break
    
    env.close()

watch_agent_play(episodes=3)

### Taxi-v3

In [ ]:
import random 
import numpy as np 
import gymnasium as gym

env = gym.make("Taxi-v3", render_mode="rgb_array")

In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n
print(f"State space: {state_space}")
print(f"Action space: {action_space}")

In [ ]:
def make_q_table(state_space, action_space):
    return np.zeros([state_space, action_space])

q_table = make_q_table(state_space, action_space)

In [ ]:
def epsilon_greedy(epsilon, q_table, state):
    if epsilon > random.random():
        return random.randint(0, action_space - 1)
    else:
        return np.argmax(q_table[state])

In [ ]:
from tqdm.auto import tqdm

def train_loop(epochs, q_table, epsilon, epsilon_decay, gamma, lr, max_steps):
    total_rewards = []

    for epoch in tqdm(range(epochs)):
        state, info = env.reset()

        for step in range(max_steps):
            move = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(move)
            q_table[state, move] += lr * (reward + gamma * np.max(q_table[next_state]) - q_table[state, move])
            state = next_state
            if done or truncated:
                break
        total_rewards.append(reward)
        epsilon = max(epsilon * epsilon_decay, 0.01)
    return q_table, total_rewards

In [ ]:
updated_q_table, reward_list = train_loop(epochs = 1000000, q_table = q_table, epsilon=1, epsilon_decay=0.99, gamma = 0.9, lr = 0.1, max_steps = 1000)

In [ ]:
np.mean(reward_list), np.std(reward_list)

In [ ]:
env = gym.make("Taxi-v3", render_mode="human")

In [ ]:
def visualize_loop(epochs, q_table):
    total_rewards = []

    for epoch in tqdm(range(epochs)):
        state, info = env.reset()

        while True:
            move = np.argmax(q_table[state])
            next_state, reward, done, truncated, info = env.step(move)
            state = next_state
            env.render()
            if done or truncated:
                break
        total_rewards.append(reward)

    env.close()
    return total_rewards

In [ ]:
visualize_loop(5, updated_q_table)

### Deep Q Learning

In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np 
import gymnasium as gym 
from collections import deque # Experience replay 
from tqdm.auto import tqdm 

In [ ]:
env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0] # 4 variables (position, velocity, angle, angular velocity)
action_size = env.action_space.n # Right or left 
print(f"State size: {state_size}")
print(f"Action size: {action_size}")

In [ ]:
class DQN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu" 

policy_net = DQN(state_size, 64, action_size).to(device) # Training network
target_net = DQN(state_size, 64, action_size).to(device) # Target network
target_net.load_state_dict(policy_net.state_dict()) # copy weights 
target_net.eval() 

optimizer = optim.Adam(policy_net.parameters(), lr=1E-3)
memory = deque(maxlen=100000)
batch_size = 32 
gamma = 0.99 
epsilon = 1.0 
epsilon_decay = 0.995 
epsilon_min = 0.01 

In [ ]:
loss_fn = nn.MSELoss()
def optimize_model(Q_values, target_Q_values):
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(Q_values, target_Q_values)
    loss.backward()
    optimizer.step()

def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        with torch.inference_mode():
            return policy_net(state).argmax().item()

def replay(iteration, target_update = 10):
    if len(memory) < batch_size:
        return

    if iteration % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())

    batch = random.sample(memory, batch_size) # random sample from memory 
    states, actions, rewards, next_states, dones = zip(*batch)
    states = torch.tensor(states, dtype=torch.float).to(device)
    actions = torch.tensor(actions, dtype=torch.long).to(device)
    rewards = torch.tensor(rewards, dtype=torch.float).to(device)
    next_states = torch.tensor(next_states, dtype=torch.float).to(device)
    dones = torch.tensor(dones, dtype=torch.bool).to(device)

    q_values = policy_net(states)
    q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q_values = target_net(next_states) 
        target = rewards + gamma * next_q_values.max(1).values * ~dones
    
    optimize_model(q_values.to(device), target.to(device))

    return states, actions, rewards, next_states, dones

In [ ]:
num_episodes = 100

for episode in range(num_episodes):
    state, info = env.reset()
    done = False
    total_reward = 0
    i = 0
    while not done:
        i += 1
        # Select action using epsilon-greedy
        action = select_action(torch.tensor(state, dtype=torch.float32).to(device), epsilon)
        
        # Take action in environment
        next_state, reward, done, _, _ = env.step(action)
        total_reward += reward
        
        # Store experience in replay memory
        memory.append((state, action, reward, next_state, done))
        
        # Move to next state
        state = next_state
        
        # Train the network
        replay(i)
    
    # Decay epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
    
    print(f"Episode {episode}, Total Reward: {total_reward}, Epsilon: {epsilon:.2f}")

In [ ]:
env = gym.make('CartPole-v1', render_mode='human')

In [ ]:
# Evaluate
policy_net.eval()
rewards_list = []

test_episodes = 3
with torch.no_grad():
    for episode in range(test_episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        i = 0
        while not done:
            i += 1
            # Select action using epsilon-greedy
            action = policy_net(torch.tensor(state, dtype=torch.float32).to(device)).argmax().item()
            
            # Take action in environment
            next_state, reward, done, _, _ = env.step(action)
            total_reward += reward
            
            # Move to next state
            state = next_state
        
        print(f"Episode {episode + 1}: Total Reward = {total_reward}, Steps = {i}")
        rewards_list.append(total_reward)

### Deep Q Learning Example 2: 
#### Deep convolution Q learning network for Breakout-v5, fun pong game 

In [ ]:
!pip install -qqq "gymnasium[atari]"

In [ ]:
import gymnasium as gym 
import torch 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
import numpy as np 
import random 
from collections import deque
import ale_py 
from torchvision.transforms import v2 
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation

In [ ]:
gym.register_envs(ale_py)
env = gym.make("ALE/Breakout-v5", render_mode = "rgb_array")
# Process like grayscale, resize, skip frames
env = AtariPreprocessing(env, frame_skip = 1, scale_obs=True)
# Stack the frames 
env = FrameStackObservation(env, stack_size = 4)

In [ ]:
state_size = env.observation_space.shape # deal with this shape later make sure to convert to greyscale and stuff 
action_size = env.action_space.n
print(state_size, action_size)

In [ ]:
''' 
Goal is to stack 4 frames together -> Get context of many frames together 
'''
TIME_CHANNELS = 4

class DQN(nn.Module):
    # (4, 84, 84)
    def __init__(self, input_size, hidden_size, output_size):
        super(DQN, self).__init__()
        self.convs = nn.Sequential(
            # (B, C, H, W)
            nn.Conv2d(TIME_CHANNELS, 32, kernel_size=8, stride = 4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size = 4, stride = 2),
            nn.ReLU(),
            nn.Conv2d(64,64, kernel_size = 3, stride = 1),
            nn.ReLU()
        )

        ''' 
        conv_out_size = self.get_conv_shape(input_size)
        Then put into the linear layer
        '''

        self.fc1 = nn.Linear(3136, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
    
    '''
    Get's the shape of the flatten tensor if im feeling lazy 
    def get_conv_shape(self, shape):
        dummy_output = self.conv(torch.zeros(1, *shape))
        return int(torch.prodd(torch.tensor(o.shape))) # torch.prod multiplies the numbers in the tensor together
    '''
    
    def forward(self, x: torch.tensor):
        B = x.shape[0]
        x = self.convs(x).view(B, -1) # flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

INPUT_SHAPE = (1,4,84,84)
device = "mps" if torch.backends.mps.is_available() else "cpu"
predictor_model = DQN(INPUT_SHAPE, 64, action_size).to(device)
target_model = DQN(INPUT_SHAPE, 64, action_size).to(device)
optimizer = optim.Adam(predictor_model.parameters(), lr=0.001)

# Copy weights from predictor to target
target_model.load_state_dict(predictor_model.state_dict())
target_model.eval()
device

In [ ]:
rand_tensor = torch.rand((1,4,84,84)).to(device)
predictor_model(rand_tensor).shape

In [ ]:
from inspect import _void

memory = deque(maxlen=1000000)
'''
memory contains
(state, action, reward, next_state, done)
'''
EPOCHS = 1000
BATCH_SIZE = 32
GAMMA = 0.99 
LR = 1E-4 
EPSILON = 0.5 
EPSILON_DECAY = 0.90
EPSILON_MIN = 0.01 
UPDATE_ITERATIONS = 10 
MAX_STEPS = 1000

loss_fn = nn.MSELoss()

# Needed without the frame stacking and transforms
transforms = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(),
    v2.ToDtype(torch.float32, scale = True),
])

def greedy_step(state, epsilon = EPSILON) -> int:
    if random.random() < epsilon:
        return random.randint(0, action_size - 1)  
    else:
        return predictor_model(state).argmax().item()

def optimize_model(predicted_q_value, ideal_q_value) -> _void:
    loss = loss_fn(predicted_q_value, ideal_q_value)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

def replay(iteration, updated_iterations = UPDATE_ITERATIONS, batch_size = BATCH_SIZE, gamma = GAMMA) -> _void:
    if len(memory) < BATCH_SIZE:
        return 
    
    if iteration % updated_iterations == 0:
        target_model.load_state_dict(predictor_model.state_dict())
    # Sample some from the memory and test out our values 
    batch = random.sample(memory, batch_size)
    states, actions, rewards, next_states, done = zip(*batch)
    
    states = torch.tensor(states, dtype = torch.float).to(device)
    actions = torch.tensor(actions, dtype=torch.long).to(device) # remember actions have to be long 
    rewards = torch.tensor(rewards, dtype=torch.float).to(device)
    next_states = torch.tensor(next_states, dtype = torch.float).to(device)
    dones = torch.tensor(done, dtype=torch.bool).to(device) # bools 
    # predict the best actions
    q_values = predictor_model(states)
    # get every action taken 
    q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        target_values = target_model(next_states)
        target = rewards + gamma * target_values.max(1).values * ~dones 

    optimize_model(q_values, target)

In [ ]:
from tqdm.auto import tqdm 

reward_log = []

for epoch in tqdm(range(EPOCHS)):
    total_reward = 0
    state, info = env.reset()
    i = 0
    while True:
        i += 1
        stacked_state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)  # [1,4,H,W]
        action = greedy_step(stacked_state_tensor, EPSILON)

        # Get next states
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

        memory.append((
            state, 
            action, 
            reward, 
            next_state, 
            done
            ))
        replay(i, gamma=GAMMA)  # Pass gamma parameter

        state = next_state 

        if done:
            break
    
    reward_log.append(total_reward)
    print("REWARD: ", total_reward)
    EPSILON = max(EPSILON * EPSILON_DECAY, EPSILON_MIN) 

### Policy Gradient: using CartPole

<img src = "/Users/ryanchen/Desktop/Python/AI/RL/Screenshot 2026-03-10 at 10.18.32 PM.png"/>

<h5>
    <code>
    Can't calcualate the probability of getting into the next state as that's part of the environment and we don't know about that. So we approximate the gradient.
    </code>
</h5>

In [12]:
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
from tqdm.auto import tqdm 

device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [13]:
env = gym.make("CartPole-v1", render_mode = "rgb_array")
eval_env = gym.make("CartPole-v1", render_mode = "human")

In [14]:
s_size = env.observation_space.shape[0]
a_size = env.action_space.n
s_size, a_size

(4, np.int64(2))

<img src = "/Users/ryanchen/Desktop/Python/AI/RL/Screenshot 2026-03-11 at 4.20.14 PM.png"/>

In [82]:
class NeuralNetPolicy(nn.Module):
    def __init__(self, state_size = s_size, action_size = a_size, hidden_size = 64):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size)
        )
    def forward(self, x: torch.tensor):
        x = self.layers(x)
        # Probability for moving left/right
        return F.softmax(x, dim = 1)
    
    def act(self, state: np.array):
       state = torch.from_numpy(state).float().unsqueeze(0).to(device)
       probs = self.forward(state)
       # Sample from probability distribution 
       m = Categorical(probs)
       action = m.sample()
       return action.item(), m.log_prob(action) # calculate log probability of chosen aciton 

model = NeuralNetPolicy().to(device)

In [ ]:
'''
So this is the formula that's used, specifically for EVERY STATE/ACTION that goes on in an episode 
that's why we use the loops. 
It's cuz we calcualte log probability and discounted rewards from rewards for every state
'''
def REINFORCE(model, optimizer, epochs, max_t, gamma):
    scores_deque = deque(maxlen = 100)
    scores = []
    for episode in tqdm(range(1, epochs)):
        saved_log_probs= []
        rewards = []
        state, info = env.reset()
        # Max timesteps to iterate over 
        for t in range(max_t):
            # Model choses action
            action, log_prob = model.act(state)
            saved_log_probs.append(log_prob)
            # Take step 
            state, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)

            done = terminated or truncated
            if done:
                break
        # Collect scores after done 
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        returns = deque(maxlen = max_t)
        n_steps = len(rewards)
        ''' 
        “Oh, it makes sense because for Q-learning, you're kind of learning on the spot. 
        So every time you kind of take a step, the difference is gonna be the reward at that specific step plus gamma times the maximum Q-value at the next state that you're in. 
        But for this one, because it's episodic thing, you calculate the discounted rewards starting from every state inside the episode. 
        So that makes sense.” - Ryan
        '''
        # Use recursion to calcualate discounted rewards starting from last -> first step
        for t in reversed(range(n_steps)):
            disc_return_t = returns[0] if len(returns) > 0 else 0 
            # Since we reversed, we append left 
            returns.appendleft(gamma * disc_return_t + rewards[t])

        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad(set_to_none = True)
        policy_loss.backward()
        optimizer.step()

        if episode % 50 == 0:
            print(f"Episode: {episode} | Average Score: {np.mean(scores_deque):.2f}")
        
    return scores 

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 1E-2)
scores = REINFORCE(
    model = model,
    optimizer = optimizer,
    epochs = 500,
    max_t = 1000,
    gamma = 0.99,
)

  0%|          | 0/999 [00:00<?, ?it/s]

Episode: 50 | Average Score: 56.56
Episode: 100 | Average Score: 62.75
Episode: 150 | Average Score: 71.24
Episode: 200 | Average Score: 86.90
Episode: 250 | Average Score: 63.41
Episode: 300 | Average Score: 38.64
Episode: 350 | Average Score: 143.53
Episode: 400 | Average Score: 228.26
Episode: 450 | Average Score: 164.52
Episode: 500 | Average Score: 103.59
Episode: 550 | Average Score: 126.63
Episode: 600 | Average Score: 149.72
Episode: 650 | Average Score: 147.05
Episode: 700 | Average Score: 141.51
Episode: 750 | Average Score: 125.53
Episode: 800 | Average Score: 108.34
Episode: 850 | Average Score: 105.12


In [48]:
### Evaluate 
model.eval()
eval_episodes = 10

with torch.no_grad():
    for episode in range(eval_episodes):
        state, info = eval_env.reset()
        reward_list = []
        while True:
            action, log_prob = model.act(state)
            state, reward, terminated, truncated, _ = eval_env.step(action)
            reward_list.append(reward)

            done = terminated or truncated
            eval_env.render()
            if done:
                print(f"FINISHED | Reward: {sum(reward_list)}")
                break
    


FINISHED | Reward: 9.0
FINISHED | Reward: 10.0
FINISHED | Reward: 10.0
FINISHED | Reward: 9.0
FINISHED | Reward: 9.0
FINISHED | Reward: 11.0
FINISHED | Reward: 9.0
FINISHED | Reward: 9.0
FINISHED | Reward: 8.0
FINISHED | Reward: 9.0
